# DLP Insider Threat Detection â€” Full Pipeline

**Before running:**
1. Set Runtime â†’ Change runtime type â†’ **T4 GPU**
2. Get your Kaggle API key: kaggle.com â†’ Account â†’ API â†’ *Create New Token* â†’ downloads `kaggle.json`
3. Run all cells top to bottom

**Pipeline steps**
| Step | What happens |
|------|--------------|
| 1 | Clone repo from GitHub |
| 2 | Install dependencies |
| 3 | Upload Kaggle credentials |
| 4 | Download CERT dataset from Kaggle |
| 5 | Configure paths |
| 6 | Clean all raw data |
| 7 | Train Isolation Forest (baseline) |
| 8 | Visualize Isolation Forest results |
| 9 | Train LSTM Autoencoder (GPU) |
| 10 | Visualize LSTM results |
| 11 | (Optional) Save outputs to Google Drive |

## Step 1 â€” Clone repo from GitHub

In [ ]:
import os

REPO_URL = 'https://github.com/Taha-coder-star/DLP-PROJECt.git'
REPO_DIR = '/content/dlp-project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

## Step 2 â€” Install dependencies

In [ ]:
!pip install -r requirements.txt kaggle -q
print('Done.')

## Step 3 â€” Upload Kaggle credentials
Go to **kaggle.com â†’ Account â†’ API â†’ Create New Token** to download `kaggle.json`, then run this cell and upload it.

In [ ]:
from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()  # select kaggle.json from your computer

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured.')

## Step 4 â€” Download CERT dataset from Kaggle
Find the dataset slug on Kaggle: open the dataset page â†’ look at the URL â†’ `kaggle.com/datasets/USERNAME/DATASET-NAME`.
Set `KAGGLE_DATASET` below to `USERNAME/DATASET-NAME`.

In [ ]:
# ===================== UPDATED v4 =====================
KAGGLE_DATASET = 'mrajaxnp/cert-insider-threat-detection-research'
ARCHIVE_DIR    = '/content/dlp-data/archive'

import os, zipfile
from pathlib import Path
os.makedirs(ARCHIVE_DIR, exist_ok=True)

REQUIRED_FILES = [
    'email.csv', 'psychometric.csv', 'logon.csv',
    'device.csv', 'file.csv', 'decoy_file.csv', 'users.csv',
]

archive = Path(ARCHIVE_DIR)

for fname in REQUIRED_FILES:
    dest    = archive / fname
    zip_path = archive / (fname + '.zip')

    # unzip if zip is present but CSV is missing
    if zip_path.exists() and not dest.exists():
        print(f'  Unzipping {zip_path.name} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(ARCHIVE_DIR)
        zip_path.unlink()

    if dest.exists():
        print(f'  {fname} already present.')
        continue

    # download (will save as fname.zip), then unzip
    print(f'  Downloading {fname} ...')
    !kaggle datasets download -d {KAGGLE_DATASET} -f {fname} -p {ARCHIVE_DIR} --force
    if zip_path.exists():
        print(f'  Unzipping {zip_path.name} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(ARCHIVE_DIR)
        zip_path.unlink()

# verify
found = sorted(archive.glob('*.csv'))
print(f'\nFound {len(found)} CSV files:')
for f in found:
    print(f'  {f.name:40s} {f.stat().st_size / 1_048_576:8.1f} MB')

missing = set(REQUIRED_FILES) - {f.name for f in found}
if missing:
    print('WARNING: missing files:', missing)
else:
    print('All required files present.')

## Step 5 â€” Configure paths

In [ ]:
import os
from pathlib import Path

# All data lives in Colab local storage for this session
os.environ['DLP_ROOT'] = '/content/dlp-data'

root = Path(os.environ['DLP_ROOT'])
for d in ['archive', 'cleaned', 'models', 'plots']:
    (root / d).mkdir(parents=True, exist_ok=True)

print('DLP_ROOT :', os.environ['DLP_ROOT'])
print('GPU      :', end=' ')
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT available â€” go to Runtime > Change runtime type > T4 GPU')

## Step 6 â€” Clean all raw data
> Takes ~10â€“20 min (processes ~10M rows across all files).

In [ ]:
!python scripts/clean_cert_email_data.py

## Step 7 â€” Train Isolation Forest (baseline)

In [ ]:
!python colab/train_isolation_forest_cert.py

## Step 8 â€” Visualize Isolation Forest results

In [ ]:
!python colab/visualize_isolation_forest_cert.py

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import os

for img in sorted((Path(os.environ['DLP_ROOT']) / 'plots').glob('iforest_*.png')):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(img))
    ax.axis('off')
    ax.set_title(img.stem)
    plt.tight_layout()
    plt.show()

## Step 9 â€” Train LSTM Autoencoder (GPU)
> Trains one model per user (1,000 models). Takes ~30â€“60 min on T4 GPU.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU found! Go to Runtime > Change runtime type > T4 GPU and re-run from Step 5.'
print('GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
!python colab/train_lstm_autoencoder_cert.py

## Step 10 â€” Visualize LSTM results

In [ ]:
!python colab/visualize_lstm_autoencoder_cert.py

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import os

for img in sorted((Path(os.environ['DLP_ROOT']) / 'plots').glob('lstm_*.png')):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(mpimg.imread(img))
    ax.axis('off')
    ax.set_title(img.stem)
    plt.tight_layout()
    plt.show()

## Step 11 — Evaluate against ground truth
Scores both models against the 70 known CERT r4.2 insiders and writes
`models/evaluation_report.json` — needed by the **Ground Truth** tab
in the monitoring dashboard. `insiders.csv` is already bundled in the repo.

In [ ]:
!python colab/evaluate_cert.py

## Step 11b — Threshold Analysis
Evaluates saved scores at percentile thresholds (90 / 95 / 97 / 99). No retraining — reads directly from scored CSV files.

In [ ]:
!python colab/threshold_analysis.py

## Step 11c — Top-K Ranking Analysis
Ranks all users by max anomaly score and evaluates Precision / Recall / F1 for K = 10, 20, 50, 100. No retraining.

In [ ]:
!python colab/topk_analysis.py

## Step 11d — Filter-then-Rank Analysis
Filters users by a percentile threshold (95th / 97th), then ranks survivors by score and evaluates top K = 10, 20, 50. No retraining.

In [ ]:
!python colab/filter_topk_analysis.py

## Step 11e -- Fixed User-Level Evaluation
Correct pipeline: aggregate daily scores to one score per user (max/mean/p95),
compute percentile thresholds from train user scores (not train rows), then
filter and rank at user level. Compares all aggregations and summarises the best.

In [ ]:
!python colab/user_level_eval.py

## Step 11f -- Weighted Risk Scoring + Visualizations
Runs the AI risk scorer (weighted evidence aggregation) and generates
6 presentation-ready plots saved to plots/user_level/.

In [ ]:
import os, sys
# DLP_ROOT is already set by Step 5; DLP_REPO defaults to /content/dlp-project
os.makedirs("plots/user_level", exist_ok=True)
!python colab/visualize_user_level.py
print("Plots saved to plots/user_level/")

## Step 12 -- Launch UEBA Dashboard
Launches the UEBA insider-threat dashboard via Streamlit + ngrok public tunnel.
Open the printed URL in your browser. No Kaggle token needed for this step.

In [ ]:
# ── Launch UEBA Dashboard ──────────────────────────────────────────────
!pip install streamlit pyngrok -q

import threading, time, os
from pyngrok import ngrok

ngrok.kill()  # close any previous tunnel

def run_streamlit():
    import subprocess
    env = os.environ.copy()
    env["DLP_REPO"] = "/content/dlp-project"
    subprocess.run([
        "streamlit", "run", "app/ueba_dashboard.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
    ], env=env)

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(6)  # wait for Streamlit to start

tunnel = ngrok.connect(8501)
print("=" * 55)
print("UEBA Dashboard URL:", tunnel.public_url)
print("=" * 55)
print("Open the URL above in your browser.")
print("Sidebar: choose model / aggregation / threshold / K")
print("Click Run Analysis to see results and charts.")